# 🌿 CropGuard AI — PlantVillage Multi-Model Training (Kaggle GPU)

Train **MobileNetV2** (and optionally **ResNet50**, **InceptionV3**, **DenseNet121**, **VGG16**) on the PlantVillage dataset using Kaggle's free GPU.

### Before Running:
1. **Add Dataset:** Click **+ Add Input** (top-right) → **Datasets** tab → search `abdallahalidev/plantvillage-dataset` → click **+**. *(Or turn Internet On and it downloads automatically.)*
2. **Enable GPU:** Right sidebar → **Session options** → **Accelerator** → select **GPU T4 x2** (or **GPU P100**).
3. **Turn on Internet:** **Session options** → **Internet on**.
4. **IMPORTANT — Use Background Commit instead of Draft Run:** Click **Save Version** (top-right) → **Save & Run All (Commit)** → **Save**. That keeps Kaggle running safely in the cloud even if your browser sleeps or disconnects, and permanently saves your Output files!

In [ ]:
# Configuration: start with MobileNetV2 (~10 min on GPU, ~15 MB file).
# To train and benchmark all 5 models for your report (~1.5 hrs on GPU), use:
# MODELS_TO_TRAIN = ["MobileNetV2", "DenseNet121", "ResNet50", "InceptionV3", "VGG16"]
MODELS_TO_TRAIN = ["MobileNetV2"]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
SEED = 42
OUTPUT_DIR = "/kaggle/working/models"

In [ ]:
import glob
import json
import os
import shutil
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import DenseNet121, InceptionV3, MobileNetV2, ResNet50, VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

In [ ]:
# Automatically locate the PlantVillage 'color' directory inside /kaggle/input (or download via kagglehub)
def find_dataset_dir(root_dir):
    if not os.path.exists(root_dir):
        return None
    matches = []
    for dirpath, dirnames, _ in os.walk(root_dir, followlinks=True):
        if any(name in dirnames for name in ("Apple___Apple_scab", "Potato___Early_blight", "Tomato_Early_blight")):
            matches.append(dirpath)
    for m in matches:
        if os.path.basename(m).lower() == "color":
            return m
    for m in matches:
        if "color" in m.lower() or os.path.basename(m).lower() == "train":
            return m
    return matches[0] if matches else None

print("Contents of /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "missing")
DATASET_DIR = find_dataset_dir("/kaggle/input")

if not DATASET_DIR:
    print("Dataset not found in /kaggle/input — downloading via kagglehub (this takes ~20s)...")
    import kagglehub
    dl_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
    DATASET_DIR = find_dataset_dir(dl_path)

if not DATASET_DIR:
    raise FileNotFoundError("Could not locate PlantVillage color images folder.")

class_names = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
print(f"Dataset found: {DATASET_DIR}")
print(f"Total classes: {len(class_names)}")

In [ ]:
# Build an exact stratified 80% train / 10% validation / 10% test split
filepaths, labels = [], []
valid_exts = {".jpg", ".jpeg", ".png"}

for cls in class_names:
    cls_dir = os.path.join(DATASET_DIR, cls)
    for fname in os.listdir(cls_dir):
        if os.path.splitext(fname)[1].lower() in valid_exts:
            filepaths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

df = pd.DataFrame({"filepath": filepaths, "label": labels})
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Fast parallel tf.data pipeline (3-5x faster on GPU than ImageDataGenerator)
label_to_idx = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

def load_and_preprocess(path, label, augment=False):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if augment:
        img = tf.image.random_flip_left_right(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
        img = tf.image.random_brightness(img, max_delta=0.1)
    return img, label

def make_dataset(split_df, augment=False, shuffle=False):
    paths = split_df["filepath"].values
    ys = tf.one_hot([label_to_idx[l] for l in split_df["label"].values], depth=num_classes)
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(split_df), seed=SEED)
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, augment=augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, augment=True, shuffle=True)
val_ds = make_dataset(val_df, augment=False, shuffle=False)
test_ds = make_dataset(test_df, augment=False, shuffle=False)
test_labels = np.array([label_to_idx[l] for l in test_df["label"].values])

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

MODEL_REGISTRY = {
    "MobileNetV2": MobileNetV2,
    "ResNet50": ResNet50,
    "InceptionV3": InceptionV3,
    "DenseNet121": DenseNet121,
    "VGG16": VGG16,
}

def build_model(model_name, num_classes):
    base_model = MODEL_REGISTRY[model_name](
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    for layer in base_model.layers:
        layer.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

comparison = {}

for model_name in MODELS_TO_TRAIN:
    print(f"\n{'=' * 60}\nTraining {model_name}\n{'=' * 60}")
    model = build_model(model_name, len(class_names))
    model_path = os.path.join(OUTPUT_DIR, f"{model_name}_plant_disease.h5")
    callbacks = [
        ModelCheckpoint(filepath=model_path, monitor="val_loss", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]

    t0 = time.time()
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1
    )
    training_time = round(time.time() - t0, 2)

    if not os.path.exists(model_path):
        model.save(model_path)
    size_mb = round(os.path.getsize(model_path) / (1024 * 1024), 2)

    # Training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history["accuracy"], label="Train")
    ax1.plot(history.history["val_accuracy"], label="Val")
    ax1.set_title(f"{model_name} - Accuracy")
    ax1.legend(); ax1.grid(True)
    ax2.plot(history.history["loss"], label="Train")
    ax2.plot(history.history["val_loss"], label="Val")
    ax2.set_title(f"{model_name} - Loss")
    ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name}_training_history.png"), dpi=150)
    plt.close(fig)

    # Test evaluation
    y_pred_probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_labels

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    with open(os.path.join(OUTPUT_DIR, f"{model_name}_classification_report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(18, 16))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{model_name} - Confusion Matrix")
    plt.xticks(rotation=90, fontsize=6); plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name}_confusion_matrix.png"), dpi=150)
    plt.close(fig)

    comparison[model_name] = {
        "accuracy": round(float(acc), 4),
        "precision_macro": round(float(prec), 4),
        "recall_macro": round(float(rec), 4),
        "f1_macro": round(float(f1), 4),
        "training_time_sec": training_time,
        "model_file_size_mb": size_mb,
    }
    with open(os.path.join(OUTPUT_DIR, "model_comparison.json"), "w") as f:
        json.dump(comparison, f, indent=2)
    print(f"\n✅ {model_name}: Acc={acc:.4f}, F1={f1:.4f}, Size={size_mb} MB, Time={training_time}s")

    del model
    tf.keras.backend.clear_session()

zip_path = shutil.make_archive("/kaggle/working/trained_models", "zip", OUTPUT_DIR)
print(f"\n🎉 Done! Download your trained artefacts from: {zip_path}")